# 📈 Stock Breakout Scanner with Risk Management (SL, T1, T2)

**Original Strategy:** YouTuber's Moving Average Breakout + CAR Confirmation  
**Enhanced With:** Stop Loss, Target Prices, and Risk Management Levels

**What This Does:**
- Downloads 2 years of stock data from Yahoo Finance
- Calculates Moving Averages (30, 50, 200 day)
- Checks if price is above all three DMAs (trend confirmation)
- Calculates CAR to ensure trend is getting stronger
- **NEW:** Calculates Stop Loss, Entry, T1, T2 (risk management)
- **NEW:** Computes Risk, Reward, and Risk:Reward Ratio
- Ranks stocks by quality
- Exports results to Excel

## Step 1: Import Required Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import warnings
import logging
from datetime import datetime

# Suppress Yahoo Finance warnings and unnecessary logs for cleaner output
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

## Step 2: Define the Main Scanner Function with Risk Management

In [ ]:
def advanced_stock_scanner(ticker_list):
    """
    Scans stocks for breakout signals with automatic risk management levels.
    
    Parameters:
    -----------
    ticker_list : list
        List of stock tickers (NSE format with .NS suffix)
    
    Returns:
    --------
    pd.DataFrame : Results with price levels, risk management, and targets
    """
    
    results = []
    today_date = datetime.now().strftime("%d-%m-%Y")
    
    print(f"🔍 Scanning {len(ticker_list)} stocks... Please wait.\n")
    
    for ticker in ticker_list:
        try:
            # Download 2 years of daily price data
            data = yf.download(ticker, period="2y", interval="1d", progress=False)
            
            # Skip stocks with insufficient data
            if data.empty or len(data) < 200:
                continue
            
            # Extract closing prices
            close_prices = data['Close'].squeeze()
            
            # Calculate Moving Averages
            dma_30 = close_prices.rolling(window=30).mean().iloc[-1]
            dma_50 = close_prices.rolling(window=50).mean().iloc[-1]
            dma_200 = close_prices.rolling(window=200).mean().iloc[-1]
            
            # Get today's closing price (Current Market Price)
            cmp = close_prices.iloc[-1]
            
            # Calculate distance from 200 DMA (how far above the trend line)
            dist_200_dma = ((cmp - dma_200) / dma_200) * 100
            
            # Find the highest high in the last year (252 trading days)
            last_1y_data = data.tail(252)
            high_date = last_1y_data['High'].squeeze().idxmax()
            
            # Extract closing prices from high date onwards
            car_data = close_prices.loc[high_date:]
            
            # Skip if insufficient CAR data
            if len(car_data) < 10:
                continue
            
            # Calculate Cumulative Average Return
            car_values = car_data.expanding().mean()
            
            # Get last 10 days of CAR values
            last_10_car = car_values.tail(10)
            
            # Check if CAR is monotonically increasing (trend getting stronger)
            if last_10_car.is_monotonic_increasing:
                car_status = 'Positive'
            else:
                car_status = 'Negative'
            
            # YOUTUBER'S CORE FILTERS
            # Condition 1: Price > 30 DMA
            # Condition 2: Price > 50 DMA
            # Condition 3: Price > 200 DMA
            # Condition 4: CAR is positive (trend confirmed)
            if not ((cmp > dma_30) and (cmp > dma_50) and (cmp > dma_200) and (car_status == 'Positive')):
                continue
            
            # ============================================================
            # RISK MANAGEMENT CALCULATIONS (NEW)
            # ============================================================
            
            # Stop Loss: Minimum of 10-day swing low and 30-DMA
            # This provides both technical support and moving average backup
            swing_low = data['Low'].tail(10).min()
            sl = min(swing_low, dma_30)
            
            # Entry Price: Current Market Price
            entry = cmp
            
            # Risk: Distance from entry to stop loss
            risk = entry - sl
            
            # Skip if risk calculation is invalid (negative risk)
            if risk <= 0:
                continue
            
            # Target Prices based on Risk:Reward multiples
            # T1: Entry + (2 × Risk) - First profit-taking level
            # T2: Entry + (3 × Risk) - Main profit target
            # T3: Entry + (5 × Risk) - Extended upside target
            t1 = entry + (2 * risk)
            t2 = entry + (3 * risk)
            t3 = entry + (5 * risk)
            
            # Calculate Reward percentage (based on T2)
            reward_percent = ((t2 - entry) / entry) * 100
            
            # Calculate Stop Loss percentage
            sl_percent = ((entry - sl) / entry) * 100
            
            # Calculate Risk:Reward Ratio
            # This shows how much profit we make for each unit of risk
            # Example: 3:1 means we make 3 rupees for every 1 rupee at risk
            if sl_percent > 0:
                rr_ratio = reward_percent / sl_percent
            else:
                rr_ratio = 0
            
            # 52-Week High Distance
            high_52 = data['High'].tail(252).max()
            dist_52 = ((high_52 - entry) / high_52) * 100
            
            # 1-Month Return
            returns = ((entry - close_prices.iloc[-21]) / close_prices.iloc[-21]) * 100 if len(close_prices) > 21 else 0
            
            # Trend Strength
            if dma_30 > dma_50 > dma_200:
                trend = "Strong Uptrend"
            else:
                trend = "Weak"
            
            # Signal Action
            action = '🟢 Positive Breakout'
            
            # Store the results
            results.append({
                'Date': today_date,
                'Stock': ticker.replace('.NS', ''),
                'CMP': round(entry, 2),
                '30 DMA': round(dma_30, 2),
                '50 DMA': round(dma_50, 2),
                '200 DMA': round(dma_200, 2),
                '200 DMA Dist %': round(dist_200_dma, 2),
                '52W High Dist %': round(dist_52, 2),
                'SL': round(sl, 2),
                'SL %': round(sl_percent, 2),
                'T1': round(t1, 2),
                'T2': round(t2, 2),
                'T3': round(t3, 2),
                'Risk': round(risk, 2),
                'Reward %': round(reward_percent, 2),
                'R:R Ratio': round(rr_ratio, 2),
                '1M Return %': round(returns, 2),
                'Trend': trend,
                'CAR Status': car_status,
                'Action': action
            })
        
        except Exception as e:
            # Skip stocks that fail to download or process
            pass
    
    # Create DataFrame and sort by 200 DMA distance (ascending)
    if results:
        df_positive = pd.DataFrame(results)
        df_positive = df_positive.sort_values(by='200 DMA Dist %', ascending=True)
        return df_positive
    else:
        return pd.DataFrame()

## Step 3: Define the Stock List (210 NSE Stocks)

In [ ]:
# Comprehensive list of major NSE stocks to scan
my_stocks = [
    '360ONE.NS', 'ABB.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'ADANIENSOL.NS',
    'ADANIENT.NS', 'ADANIGREEN.NS', 'ADANIPORTS.NS', 'ADANIPOWER.NS', 'ABCAPITAL.NS',
    'ALKEM.NS', 'AMBER.NS', 'AMBUJACEM.NS', 'ANGELONE.NS', 'APOLLOHOSP.NS',
    'ASHOKLEY.NS', 'ASIANPAINT.NS', 'ASTRAL.NS', 'AUROPHARMA.NS', 'DMART.NS',
    'AXISBANK.NS', 'BSE.NS', 'BAJAJ-AUTO.NS', 'BAJFINANCE.NS', 'BAJAJFINSV.NS',
    'BAJAJHLDNG.NS', 'BANDHANBNK.NS', 'BANKBARODA.NS', 'BANKINDIA.NS', 'BDL.NS',
    'BEL.NS', 'BHARATFORG.NS', 'BHEL.NS', 'BPCL.NS', 'BHARTIARTL.NS',
    'BIOCON.NS', 'BLUESTARCO.NS', 'BOSCHLTD.NS', 'BRITANNIA.NS', 'CGPOWER.NS',
    'CANBK.NS', 'CDSL.NS', 'CHOLAFIN.NS', 'CIPLA.NS', 'COALINDIA.NS',
    'COCHINSHIP.NS', 'COFORGE.NS', 'COLPAL.NS', 'CAMS.NS', 'CONCOR.NS',
    'CROMPTON.NS', 'CUMMINSIND.NS', 'DLF.NS', 'DABUR.NS', 'DALBHARAT.NS',
    'DELHIVERY.NS', 'DIVISLAB.NS', 'DIXON.NS', 'DRREDDY.NS', 'ETERNAL.NS',
    'EICHERMOT.NS', 'EXIDEIND.NS', 'FORCEMOT.NS', 'NYKAA.NS', 'FORTIS.NS',
    'GAIL.NS', 'GVTD.NS', 'GMRAIRPORT.NS', 'GLENMARK.NS', 'GODFRYPHLP.NS',
    'GODREJCP.NS', 'GODREJPROP.NS', 'GRASIM.NS', 'HCLTECH.NS', 'HDFCAMC.NS',
    'HDFCBANK.NS', 'HDFCLIFE.NS', 'HAVELLS.NS', 'HEROMOTOCO.NS', 'HINDALCO.NS',
    'HAL.NS', 'HINDPETRO.NS', 'HINDUNILVR.NS', 'HINDZINC.NS', 'POWERINDIA.NS',
    'HYUNDAI.NS', 'ICICIBANK.NS', 'ICICIGI.NS', 'ICICIPRULI.NS', 'IDFCFIRSTB.NS',
    'ITC.NS', 'INDIANB.NS', 'IEX.NS', 'IOC.NS', 'IRFC.NS', 'IREDA.NS',
    'INDUSTOWER.NS', 'INDUSINDBK.NS', 'NAUKRI.NS', 'INFY.NS', 'INOXWIND.NS',
    'INDIGO.NS', 'JINDALSTEL.NS', 'JSWENERGY.NS', 'JSWSTEEL.NS', 'JIOFIN.NS',
    'JUBLFOOD.NS', 'KEI.NS', 'KPITTECH.NS', 'KALYANIJIL.NS', 'KAYNES.NS',
    'KFINTECH.NS', 'KOTAKBANK.NS', 'LTF.NS', 'LICHSGFIN.NS', 'LTM.NS',
    'LT.NS', 'LAURUSLABS.NS', 'LICI.NS', 'LODHA.NS', 'LUPIN.NS',
    'MM.NS', 'MANAPPURAM.NS', 'MANKIND.NS', 'MARICO.NS', 'MARUTI.NS',
    'MFSL.NS', 'MAXHEALTH.NS', 'MAZDOCK.NS', 'MOTILALOFS.NS', 'MPHASIS.NS',
    'MCX.NS', 'MUTHOOTFIN.NS', 'NBCC.NS', 'NHPC.NS', 'NMDC.NS',
    'NTPC.NS', 'NATIONALUM.NS', 'NESTLEIND.NS', 'NAMINDIA.NS', 'NUVAMA.NS',
    'OBEROIRLTY.NS', 'ONGC.NS', 'OIL.NS', 'PAYTM.NS', 'OFSS.NS',
    'POLICYBZR.NS', 'PGEL.NS', 'PIIND.NS', 'PNBHOUSING.NS', 'PAGEIND.NS',
    'PATANJALI.NS', 'PERSISTENT.NS', 'PETRONET.NS', 'PIDILITIND.NS', 'POLYCAB.NS',
    'PFC.NS', 'POWERGRID.NS', 'PREMIERENE.NS', 'PRESTIGE.NS', 'PNB.NS',
    'RBLBANK.NS', 'RECLTD.NS', 'RADICO.NS', 'RVNL.NS', 'RELIANCE.NS',
    'SBICARD.NS', 'SBILIFE.NS', 'SHREECEM.NS', 'SRF.NS', 'MOTHERSON.NS',
    'SHRIRAMFIN.NS', 'SIEMENS.NS', 'SOLARINDS.NS', 'SONACOMS.NS', 'SBIN.NS',
    'SAIL.NS', 'SUNPHARMA.NS', 'SUPREMEIND.NS', 'SUZLON.NS', 'SWIGGY.NS',
    'TATACONSUM.NS', 'TVSMOTOR.NS', 'TCS.NS', 'TATAELXSI.NS', 'TMPV.NS',
    'TATAPOWER.NS', 'TATASTEEL.NS', 'TECHM.NS', 'FEDERALBNK.NS', 'INDHOTEL.NS',
    'PHOENIXLTD.NS', 'TITAN.NS', 'TORNTPHARM.NS', 'TRENT.NS', 'TIINDIA.NS',
    'UNOMINDA.NS', 'UPL.NS', 'ULTRACEMCO.NS', 'UNIONBANK.NS', 'UNITDSPR.NS',
    'VBL.NS', 'VEDL.NS', 'VMM.NS', 'IDEA.NS', 'VOLTAS.NS',
    'WAAREEENER.NS', 'WIPRO.NS', 'YESBANK.NS', 'ZYDUSLIFE.NS'
]

print(f"Total stocks to scan: {len(my_stocks)}")

## Step 4: Run the Scanner

In [ ]:
# Execute the scanner function
positive_breakout_data = advanced_stock_scanner(my_stocks)

## Step 5: Display Results

In [ ]:
# Display results with formatting
print("\n" + "="*150)
print("🟢 FINAL LIST: POSITIVE BREAKOUT STOCKS WITH SL, T1, T2")
print("="*150 + "\n")

if positive_breakout_data.empty:
    print("❌ No stocks matched the criteria today.\n")
else:
    # Display key columns
    display_cols = ['Stock', 'CMP', '30 DMA', '50 DMA', '200 DMA', 'SL', 'T1', 'T2', 'Reward %', 'R:R Ratio', 'Trend']
    print(positive_breakout_data[display_cols].to_string(index=False))
    print("\n" + "="*150)

## Step 6: Display Summary Statistics

In [ ]:
# Display detailed statistics
if not positive_breakout_data.empty:
    print("\n📊 SUMMARY STATISTICS")
    print("="*150)
    print(f"Total Stocks Found:       {len(positive_breakout_data)}")
    print(f"Average Distance from 200 DMA: {positive_breakout_data['200 DMA Dist %'].mean():.2f}%")
    print(f"Average Stop Loss %:      {positive_breakout_data['SL %'].mean():.2f}%")
    print(f"Average Reward %:         {positive_breakout_data['Reward %'].mean():.2f}%")
    print(f"Average R:R Ratio:        {positive_breakout_data['R:R Ratio'].mean():.2f}:1")
    print("="*150)

## Step 7: Export to Excel

In [ ]:
# Save complete results to Excel file
if not positive_breakout_data.empty:
    filename = "YouTuber_Breakout_Scanner_with_SL_T1_T2.xlsx"
    positive_breakout_data.to_excel(filename, index=False)
    print(f"✅ Results saved to '{filename}'")
    print(f"\nFile includes all columns:")
    for col in positive_breakout_data.columns:
        print(f"  • {col}")
else:
    print("No data to export.")

## How to Interpret the Results

### Original Columns (YouTuber's Logic):

| Column | Meaning |
|--------|----------|
| **Stock** | Stock ticker (without .NS) |
| **CMP** | Current Market Price (today's close) |
| **30/50/200 DMA** | Moving Averages |
| **200 DMA Dist %** | Distance from 200-day trend line |
| **Trend** | Strong Uptrend or Weak |
| **CAR Status** | Cumulative Average Return (Positive/Negative) |

### New Columns (Risk Management - For Your Father):

| Column | Meaning | Example |
|--------|----------|---------|
| **SL** | Stop Loss Level | ₹2720 (exit here if wrong) |
| **SL %** | Risk as % of entry | 0.71% (you risk this much) |
| **T1** | First Target | ₹2880 (take 50% profit here) |
| **T2** | Main Target | ₹2960 (take rest here) |
| **T3** | Extended Target | ₹3040 (bonus target) |
| **Risk** | Entry - SL | ₹80 (rupees at risk) |
| **Reward %** | Profit % at T2 | 5.71% (potential gain) |
| **R:R Ratio** | Risk:Reward | 3:1 (make ₹3 for every ₹1 risked) |

### Trading Plan Example (RELIANCE):

```
Entry Price: ₹2800
Stop Loss:   ₹2720 (Exit if price hits this)
T1:          ₹2880 (Take 50% profit)
T2:          ₹2960 (Take remaining 50%)

Risk:        ₹80 per share
Reward:      ₹160 to reach T2
R:R Ratio:   2:1 (good!)
```

### Quality Interpretation:

- **R:R Ratio > 2:1** = Excellent (professional trades)
- **R:R Ratio 1-2:1** = Good
- **R:R Ratio < 1:1** = Avoid (risk > reward)

- **SL % < 2%** = Tight (safe but limited room)
- **SL % 2-5%** = Ideal
- **SL % > 5%** = Wide (more room but higher risk)

### What Each Filter Means:

1. **Price > 30 DMA**: Stock has short-term momentum
2. **Price > 50 DMA**: Stock maintains medium-term consistency
3. **Price > 200 DMA**: Stock is in major long-term uptrend
4. **CAR = Positive**: Trend is getting STRONGER (not just steady)

### Trading Discipline:

✅ **DO:**
- Set Stop Loss BEFORE entering
- Take T1 profit (50%) at first target
- Trail stop loss to break-even after T1
- Take T2 profit (50%) at second target
- Keep a trading journal

❌ **DON'T:**
- Change stop loss to a lower level (hope trading)
- Hold beyond targets hoping for more
- Ignore the R:R ratio
- Trade all signals blindly
- Risk more than 1-2% per trade